# 01 — Exploration audio : waveform, STFT, spectrogrammes

Objectifs :
- Charger une paire audio `noisy` / `clean`
- Comprendre la waveform : le signal audio dans le temps
- Comprendre la STFT : passage du signal audio vers un spectrogramme
- Vérifier le round-trip `STFT -> ISTFT`
- Comprendre les `mag` : magnitudes de spectrogrammes utilisées par le modèle
- Tester le `PairedAudioDataset` PyTorch et un `DataLoader`

Ce notebook ne déplace pas les fichiers du Drive. Il lit les audios, les transforme, puis affiche des contrôles visuels pour vérifier que le pipeline est cohérent.

## Setup Colab

Objectif : rendre le code du repo importable depuis ce notebook.

Chaque notebook Colab a son propre kernel. Même si `main.ipynb` a déjà été exécuté, ce notebook doit lui aussi ajouter le dossier du projet dans `sys.path`, sinon les imports comme `from src import config` peuvent échouer.

In [ ]:
import os, sys

# Chemin du repo dans le runtime Colab.
REPO_DIR = '/content/Filtre-Voix-DL'

# On arrête tout de suite si le repo n'est pas présent : les imports src ne marcheront pas.
assert os.path.exists(REPO_DIR), (
    f'{REPO_DIR} introuvable — exécute la cellule clone de main.ipynb '
    'pour cloner/mettre à jour le repo sur ce runtime Colab.'
)

# Ajoute le repo au chemin d'import Python pour permettre `from src import ...`.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        # Recharge automatiquement les modules src/*.py si on les modifie.
        ipy.run_line_magic('load_ext', 'autoreload')
        ipy.run_line_magic('autoreload', '2')
except Exception as e:
    # Sur certaines versions de Colab, autoreload peut échouer sans bloquer le notebook.
    pass

print(f'sys.path OK — repo : {REPO_DIR}')

## Imports

Objectif : charger les outils nécessaires pour manipuler l'audio, afficher les graphes et utiliser les helpers du projet.

Les constantes viennent de `src.config` pour garder les mêmes paramètres partout : sample rate, durée fixe des clips, taille de STFT, chemins Drive, etc.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from src import config
from src import audio as A
from src.dataset import list_pairs, PairedAudioDataset

## 1. Charger une paire `noisy` / `clean`

Objectif : récupérer une paire alignée du dataset.

Une paire contient deux fichiers audio :
- `noisy` : la voix bruitée, utilisée comme entrée du futur modèle
- `clean` : la même voix sans bruit, utilisée comme cible

Le code ne copie et ne déplace aucun fichier : il liste les chemins puis lit les audios depuis le Drive.

In [ ]:
# pair_by='auto' essaie d'abord d'apparier par nom identique.
# Si aucun nom ne correspond, il apparie par ordre alphabétique trié.
pairs = list_pairs(pair_by='auto')
assert pairs, 'Aucune paire trouvée — vérifie que data/clean et data/noisy sont remplis.'
print(f'{len(pairs)} paires disponibles')

# On prend ici la première paire pour avoir un exemple stable et reproductible.
label, noisy_path, clean_path = pairs[0]
print(f'Exemple : {label}')
print(f'  noisy : {noisy_path}')
print(f'  clean : {clean_path}')

# load_audio charge en mono et rééchantillonne à config.SAMPLE_RATE, ici 16000 Hz.
noisy = A.load_audio(noisy_path)
clean = A.load_audio(clean_path)

# shape[-1] est le nombre d'échantillons audio.
# durée = nombre d'échantillons / échantillons par seconde.
print(f'noisy : shape={noisy.shape} | durée={noisy.shape[-1] / config.SAMPLE_RATE:.2f}s')
print(f'clean : shape={clean.shape} | durée={clean.shape[-1] / config.SAMPLE_RATE:.2f}s')

## Ecoute rapide

Objectif : comparer la paire à l'oreille.

Sur certains exemples, la différence entre `noisy` et `clean` est plus évidente au casque que sur les spectrogrammes, surtout si le bruit est faible.

In [ ]:
print('Noisy :')
display(Audio(noisy, rate=config.SAMPLE_RATE))

print('Clean :')
display(Audio(clean, rate=config.SAMPLE_RATE))

## 2. Waveform

Objectif : afficher le signal audio dans le temps.

Une waveform montre l'amplitude du son seconde par seconde. Elle sert surtout à vérifier la durée, les silences, les pics et l'alignement global entre `noisy` et `clean`.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

# Même axe temporel pour comparer plus facilement les deux signaux.
A.plot_waveform(noisy, title='Noisy', ax=axes[0])
A.plot_waveform(clean, title='Clean', ax=axes[1])

plt.tight_layout()
plt.show()

## 3. Spectrogrammes avec STFT

Objectif : passer d'un signal audio 1D à une représentation temps-fréquences.

La STFT découpe l'audio en petites fenêtres. Pour chaque fenêtre, elle mesure quelles fréquences sont présentes. Le spectrogramme se lit comme une image :
- axe horizontal : temps
- axe vertical : fréquences en Hz
- couleur claire : beaucoup d'énergie
- couleur sombre : peu d'énergie

In [ ]:
print(f'STFT params : n_fft={config.N_FFT}, hop={config.HOP_LENGTH}, win={config.WIN_LENGTH}')

# plot_pair affiche pour chaque audio : waveform + spectrogramme en dB.
A.plot_pair(noisy, clean)
plt.show()

## 4. Round-trip `STFT -> ISTFT`

Objectif : vérifier que notre transformation est réversible.

Avant d'entraîner un modèle de débruitage, on veut être sûr que :

`audio -> STFT -> ISTFT -> audio reconstruit`

redonne presque le même signal. Si cette étape échoue, le problème vient du pipeline audio, pas du modèle.

In [ ]:
# Le modèle travaillera sur des clips de durée fixe : 4s = 64000 échantillons à 16 kHz.
wav = A.fix_length(clean, config.CLIP_SAMPLES, mode='center')

# STFT : signal temporel -> spectrogramme complexe [fréquences, temps].
spec = A.stft(wav)

# ISTFT : spectrogramme complexe -> signal temporel reconstruit.
# length force la sortie à avoir exactement la même longueur que wav.
recon = A.istft(spec, length=wav.shape[-1])

# Deux mesures d'erreur : la plus grosse erreur locale et l'erreur moyenne globale.
err = np.abs(wav - recon).max()
rms = np.sqrt(np.mean((wav - recon) ** 2))

print(f'spec shape  : {spec.shape}  (freq, time)')
print(f'wav  shape  : {wav.shape}')
print(f'erreur max  : {err:.2e}')
print(f'erreur RMS  : {rms:.2e}')

# Si l'erreur est supérieure à 1e-3, la reconstruction est trop éloignée.
assert err < 1e-3, 'Reconstruction trop éloignée — vérifier les params STFT.'

## 5. Tester `PairedAudioDataset`

Objectif : vérifier que le dataset PyTorch renvoie bien les données dans le format attendu pour l'entraînement.

Avec `return_spectrogram=True`, chaque exemple contient :
- `noisy_mag` : magnitude du spectrogramme bruité, future entrée du modèle
- `clean_mag` : magnitude du spectrogramme propre, future cible
- `noisy_phase` : phase du noisy, conservée pour reconstruire un audio après prédiction

In [ ]:
from torch.utils.data import DataLoader

# crop_mode='random' prend un extrait aléatoire de 4s quand le clip est trop long.
# C'est utile à l'entraînement pour varier les extraits vus par le modèle.
ds = PairedAudioDataset(return_spectrogram=True, crop_mode='random', pair_by='auto')
print(f'Taille dataset : {len(ds)}')

# Un sample = une seule paire noisy/clean transformée en spectrogrammes.
sample = ds[0]
for k, v in sample.items():
    print(f'  {k:14s} : {type(v).__name__}', getattr(v, 'shape', v))

## 6. Tester le `DataLoader`

Objectif : vérifier que PyTorch peut regrouper plusieurs exemples en un batch.

Le `batch_size=4` est arbitraire : il veut dire que le DataLoader retourne 4 paires à la fois. On peut augmenter cette valeur pendant l'entraînement si la mémoire GPU/RAM le permet. Ici, `4` est surtout pratique pour tester les dimensions.

In [ ]:
loader = DataLoader(ds, batch_size=4, shuffle=True, num_workers=0)

# iter(loader) crée un itérateur, next(...) récupère le premier batch.
batch = next(iter(loader))

print('Batch :')
for k, v in batch.items():
    if hasattr(v, 'shape'):
        # Exemple attendu pour noisy_mag : [4, 1, fréquences, temps].
        # 4 = nombre d'exemples dans le batch, 1 = canal de type image noir et blanc.
        print(f'  {k:14s} : shape={tuple(v.shape)} | dtype={v.dtype}')
    else:
        print(f'  {k:14s} : {v}')

## 7. Visualiser les magnitudes `noisy_mag` et `clean_mag`

Objectif : regarder ce que le modèle verra réellement.

`mag` veut dire **magnitude**. Après la STFT, chaque case du spectrogramme est un nombre complexe. On le sépare en deux informations :
- magnitude : quantité d'énergie à une fréquence et à un instant donnés
- phase : position de l'onde, utile pour reconstruire l'audio

Le modèle apprend typiquement à transformer `noisy_mag` en `clean_mag`. La phase du noisy est gardée à part pour reconstruire un son avec `ISTFT`.

In [ ]:
import librosa

# batch['noisy_mag'] a une forme du type [batch, canal, fréquences, temps].
# [0, 0] sélectionne : premier exemple du batch, premier canal.
mag = batch['noisy_mag'][0, 0].numpy()
mag_clean = batch['clean_mag'][0, 0].numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, m, title in zip(axes, [mag, mag_clean], ['Noisy mag', 'Clean mag']):
    # Conversion en décibels : plus lisible visuellement que les amplitudes brutes.
    db = librosa.amplitude_to_db(m, ref=np.max)
    img = librosa.display.specshow(db, sr=config.SAMPLE_RATE,
                                   hop_length=config.HOP_LENGTH,
                                   x_axis='time', y_axis='hz', ax=ax)
    ax.set_title(title)
    plt.colorbar(img, ax=ax, format='%+2.0f dB')

plt.tight_layout()
plt.show()

## 8. Afficher la différence `Noisy - Clean`

Objectif : rendre les différences plus visibles.

Quand les deux spectrogrammes se ressemblent beaucoup, l'oeil voit mal le bruit. Cette cellule soustrait le spectrogramme clean au spectrogramme noisy en dB :
- rouge/orange : plus d'énergie dans `noisy`, souvent du bruit ajouté
- bleu : plus d'énergie dans `clean`
- proche de blanc : peu ou pas de différence

In [ ]:
db_noisy = librosa.amplitude_to_db(mag, ref=np.max)
db_clean = librosa.amplitude_to_db(mag_clean, ref=np.max)

# Différence positive : le noisy contient plus d'énergie à cet endroit temps/fréquence.
diff_db = db_noisy - db_clean

fig, ax = plt.subplots(figsize=(10, 4))
img = librosa.display.specshow(diff_db, sr=config.SAMPLE_RATE,
                               hop_length=config.HOP_LENGTH,
                               x_axis='time', y_axis='hz', ax=ax,
                               cmap='coolwarm')
ax.set_title('Différence Noisy - Clean (dB)')
plt.colorbar(img, ax=ax, format='%+2.0f dB')
plt.tight_layout()
plt.show()

print(f'Différence dB : min={diff_db.min():.2f} | moyenne={diff_db.mean():.2f} | max={diff_db.max():.2f}')
print("Rouge/orange : plus d'énergie dans noisy. Bleu : plus d'énergie dans clean. Blanc : proche.")

## Livrable semaine 1

Si les dernières cellules affichent les spectrogrammes et la différence `Noisy - Clean`, le pipeline minimal est valide :

`DataLoader -> batch -> noisy_mag / clean_mag / noisy_phase`

On a donc des entrées exploitables pour un futur modèle de débruitage.